In [46]:
import sys
import pandas as pd
import numpy as np
import shap
from xgboost.sklearn import XGBClassifier
from sklearn.metrics import r2_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [47]:
def create_surrogate(X, predictions, sur_parameters):
    # Create surrogate xgboost model
    surrogate = XGBClassifier(**sur_parameters)
    surrogate.fit(X, predictions)
    sur_predictions = surrogate.predict(X)
    external_fidelity = r2_score(predictions, sur_predictions)
    return surrogate, external_fidelity

def calc_shap(model, x_explain, approximate=False):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(x_explain, approximate=approximate)
    return shap_values, explainer


In [48]:
from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_breast_cancer(return_X_y=True)
X_train, _, y_train, _ = train_test_split(X, y, random_state=42)
original_model = MLPClassifier()
original_model.fit(X_train, y_train)
predictions = original_model.predict(X)

# Use agnostic explainer
sur_parameters = {'n_estimators': 100, 'max_depth': 8}
surrogate_model, external_fidelity = create_surrogate(X,
                                                predictions,
                                                sur_parameters)
# get the shapley values for the whole dataset
shap_values, explainer = calc_shap(surrogate_model, X)

The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
ntree_limit is deprecated, use `iteration_range` or model slicing instead.


[22:50:54] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.0/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [49]:
# shap.summary_plot(shap_values, X, plot_type="bar")

In [50]:
# ----------- Method 1 -----------
vals= np.abs(shap_values).mean(0)

X_train = pd.DataFrame(X_train)

feature_importance = pd.DataFrame(list(zip(X_train.columns,vals)),columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],ascending=False,inplace=True)
feature_importance["col_name"] =  'Feature ' + feature_importance["col_name"].astype(str)

print("Feature Importance - Method 1:", feature_importance.head())

Feature Importance - Method 1:       col_name  feature_importance_vals
23  Feature 23                 3.612318
13  Feature 13                 1.271456
20  Feature 20                 0.982844
10  Feature 10                 0.732769
27  Feature 27                 0.645023


In [51]:
feature_importance = feature_importance.head()

In [52]:
feature_importance.to_csv(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt', header=None, index=None, sep=' ', mode='a')

Use python code from Toyota for editing a file and add new line and text: 2nd run, 3rd run as in a Loop,
then make 2nd fopen close script from opening the txt and extracting the different values for comparison

In [110]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','a') as file:
    file.write("\nRun number: 2\n\n")
file.close()

In [157]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','r') as file:
    data = file.readlines()
file.close()

In [162]:
values = []
features = []
for i in range(5):
    for j in range(5):
        x = data[i*8+j].split()
        values.append(x[2])
        features.append(x[0]+x[1])

In [173]:
values = np.array(values)
sub_arrays = np.hsplit(values, 5)
df = pd.DataFrame(sub_arrays)
df.columns = ['Run_1','Run_2','Run_3','Run_4','Run_5']
df.index = ['1st Feature', '2nd Feature', '4rd Feature', '4th Feature', '5th Feature']

In [182]:
features = np.array(features)
sub_arrays2 = np.hsplit(features, 5)
df2 = pd.DataFrame(sub_arrays2)
df2.columns = ['Run_1','Run_2','Run_3','Run_4','Run_5']
df2.index = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']

In [187]:
df

,Run_1,Run_2,Run_3,Run_4,Run_5
1st Feature,3.219843,1.3631996,1.0051135,0.89303696,0.4313375
2nd Feature,2.8661892,1.4396228,0.7863044,0.539315,0.48167187
4rd Feature,2.5679462,1.3657334,0.9470184,0.79186934,0.48637152
4th Feature,3.631389,0.94815284,0.7561722,0.6743943,0.5441728
5th Feature,2.4727197,1.5941181,1.4069512,1.064051,0.48297918


In [11]:
import pandas as pd
cols = ["F1", "F2", "F3", "F4", "F5","F6","F7","F8","F9","F10","Class"]
data = pd.read_table(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\datasets\avila\avila-tr.txt', sep=',')
data.columns = cols

In [14]:
X = data.drop(columns='Class')
y = data["Class"]